In [1]:
# ============================================================================
# STEP 0: SSL CERT AND ENVIRONMENT
# ============================================================================
import certifi
import os
from dotenv import load_dotenv

# Set SSL certificate for HTTPS connections
os.environ["SSL_CERT_FILE"] = certifi.where()
print("SSL_CERT_FILE set to:", os.environ["SSL_CERT_FILE"])

# Load environment variables from .env
load_dotenv()


SSL_CERT_FILE set to: D:\Data_Science\CV\Lib\site-packages\certifi\cacert.pem


True

In [2]:
# Importing
import os #For env
import re
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
import datetime

In [3]:
# Init the model
llm = ChatGroq(api_key = os.getenv("GROQ_API_KEY"),
               model_name = "llama-3.3-70b-versatile",
               temperature = 0)

In [4]:
import sqlite3
# Run this first, init the database
def init_sqlite_database():
    """init from init_sqlite.sql"""
    try:
        db_path = "attendance.db"

        if not os.path.exists("init_sqlitedb.sql"):
            print("No init file")
            return None

        #init because found

        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Reading and executing the file
        with open("init_sqlitedb.sql","r") as f:
            sql_script = f.read()

        cursor.executescript(sql_script)
        conn.commit() #Saving

        cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
        tables = [table[0] for table in cursor.fetchall()]

        conn.close()
        print(f"Create SQLite db with tables: {tables}")
        return SQLDatabase.from_uri(f"sqlite:///{db_path}")
    except Exception as e:
        print("Failed to init sql db:")
        print(e)
        return None
def test_connection():
    try:
        db = SQLDatabase.from_uri("sqlite:///attendance.db")
        tables = db.get_usable_table_names()
        print(f"Connected, found: {tables}")
        
        result = db.run("SELECT COUNT(*) as student_count FROM students")
        print(f"📊 Students in database: {result}")
        return db
    except Exception as e:
        print(f"Connection failed: {e}")
        print("Creating a new sql db")
        return init_sqlite_database()

In [5]:
db = test_connection()

Connected, found: ['attendance_sessions', 'class_schedule', 'daily_attendance', 'semester_config', 'student_circumstances', 'students']
📊 Students in database: [(100,)]


In [6]:
from datetime import datetime, time
import logging

def get_connection(row_factory = None):
    """Connect to db"""
    conn = sqlite3.connect("attendance.db")
    if row_factory:
        conn.row_factory = row_factory
    return conn
def get_current_session_direct():
    """
    Get the current session, next one if it's break time
    """
    conn = get_connection()
    try:
        cursor = conn.cursor()
        query = """
            SELECT session_number, start_time, end_time
            FROM class_schedule
            WHERE start_time<=TIME('now','localtime') AND TIME('now','localtime') <=end_time
            LIMIT 1
        """
        cursor.execute(query)
        result = cursor.fetchone()
        if result is None:
            query = """
                SELECT session_number, start_time, end_time
                FROM class_schedule
                WHERE TIME('now','localtime') < start_time
                ORDER BY session_number ASC
                LIMIT 1
            """
            cursor.execute(query)
            result = cursor.fetchone()
        if result:
            # Convert string times to time
            session_number, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()

            return {
                'session_number':session_number,
                'start_time':start_time,
                'end_time':end_time
            }
        return None
    finally:
        conn.close()

In [10]:
def get_session_by_number(session_number:int):
    """Get the session by nunmber"""
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute("""
        SELECT session_number, start_time, end_time
        FROM class_schedule
        WHERE session_number = ?
        """,(session_number,))

        result = cursor.fetchone()
        if result:
            session_num, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()
            return {
                'session_number':session_num,
                'start_time':start_time,
                'end_time':end_time
            }
        return None
    finally:
        conn.close()

In [11]:
def calculate_late_minutes (entry_time, scheduled_start_time):
    """ calculate late minutes"""
    if isinstance(entry_time, datetime):
        entry_time = entry_time.time()
    if isinstance(scheduled_start_time, str):
        scheduled_start_time = datetime.strptime(scheduled_start_time, '%H:%M:%S').time()

    entry_datetime = datetime.combine(datetime.today(), entry_time)
    scheduled_datetime = datetime.combine(datetime.today(), scheduled_start_time)

    if entry_datetime > scheduled_datetime:
        delta = entry_datetime - scheduled_datetime
        return int(delta.total_seconds()/60)
    return 0


In [ ]:
def get_or_create_student(name):
    """Get or create student """
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        cursor.execute("SELECT id FROM students WHERE name = ?", (name,))
        result = cursor.fetchone()

        if not result:
            cursor.execute("INSERT INTO students (name) VALUES (?)", (name,))
            student_id = cursor.lastrowid
            conn.commit()
            logging.info(f"Created new student: {name} (ID: {student_id})")
        else:
            student_id = result[0]
            logging.info(f"Found existing student: {name} (ID: {student_id})")

        conn.close()
        return student_id
        
    except Exception as e:
        logging.error(f"Error getting/creating student: {e}")
        return None


In [48]:
def get_student_attendance_history(student_id):
    """Get the student history"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute("""
                SELECT 
                    COUNT(*) as total_sessions,
                    SUM(CASE WHEN attendance_status = 'late' THEN 1 ELSE 0 END) as late_count,
                    SUM(CASE WHEN attendance_status = 'very_late' THEN 1 ELSE 0 END) as very_late_count,
                    AVG(late_minutes) as avg_late_minutes
                FROM attendance_sessions 
                WHERE student_id = ? 
                AND session_date >= DATE('now', '-7 days')
            """, (student_id,))
            result = cursor.fetchone()
            if result:
                total,late,very_late, avg_late = result
                # Cho cac phan tu bang None trong SQLITE
                late = late or 0
                very_late = very_late or 0
                avg_late = avg_late or 0

                return f"Last 7 days: {total} sessions, {late} late, {very_late} very late, avg{avg_late:.1f} min late"
            return "No recent history"
    except Exception as e:
        logging.error(f"Error while getting student hisotry:")
        return "History unavailable"

def active_check(student_id):
    """If the date is valid, active, if not, deactive"""
    try:
        
        with get_connection() as conn:
            #deactive
            cursor = conn.cursor()
            cursor.execute("""
                SELECT id, start_date, end_date
                FROM student_circumstances 
                WHERE student_id = ? AND is_active = 1
            """, (student_id,))
            results_active = cursor.fetchall()
            cursor.execute("""
                SELECT id, start_date, end_date
                FROM student_circumstances 
                WHERE student_id = ? AND is_active = 0
            """, (student_id,))
            results_deactive = cursor.fetchall()
            for result in results_active:
                cir_id , start_date, end_date = result
                start_date = datetime.strptime(start_date, "%Y-%m-%d")
                end_date = datetime.strptime(end_date, "%Y-%m-%d")
                current_date = datetime.now()
                if current_date>end_date or current_date<start_date:
                    cursor.execute("""
                        UPDATE student_circumstances
                        SET is_active = 0
                        WHERE id = ?
                    """,(cir_id,))
            #active
            for result in results_deactive:
                cir_id , start_date, end_date = result
                start_date = datetime.strptime(start_date, "%Y-%m-%d")
                end_date = datetime.strptime(end_date, "%Y-%m-%d")
                current_date = datetime.now()
                if current_date<=end_date and current_date>=start_date:
                    cursor.execute("""
                        UPDATE student_circumstances
                        SET is_active = 1
                        WHERE id = ?
                    """,(cir_id,))       
    finally:
        conn.close()



def get_student_circumstances(student_id, session_number=None):
    """Get student circumstances with session-specific excuses"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            
            if session_number:
                # Get circumstances specific to this session
                cursor.execute("""
                    SELECT circumstance_type, description, session_numbers, excuse_type
                    FROM student_circumstances 
                    WHERE student_id = ? AND is_active = 1
                    AND date('now') BETWEEN start_date AND end_date
                    AND (session_numbers = 'all' OR session_numbers LIKE ?)
                """, (student_id, f'%{session_number}%'))
            else:
                # Get all active circumstances
                cursor.execute("""
                    SELECT circumstance_type, description, session_numbers, excuse_type
                    FROM student_circumstances 
                    WHERE student_id = ? AND is_active = 1
                    AND date('now') BETWEEN start_date AND end_date
                """, (student_id,))

            results = cursor.fetchall()
            if results:
                circumstances = []
                for circ_type, description, session_nums, excuse_type in results:
                    if session_nums and excuse_type:
                        circumstances.append(f"{circ_type}({excuse_type} for sessions:{session_nums}):{description}")
                    else:
                        circumstances.append(f"{circ_type}:{description}")
                return " | ".join(circumstances)
            return "No active circumstances"
    except Exception as e:
        logging.error(f"Error getting student circumstances: {e}")
        return "Circumstances unavailable"

In [49]:
#Test
active_check(1)

In [26]:
#Test 
get_student_attendance_history(1)
get_student_circumstances(1,1)

'transportation(late_arrival for sessions:1):Bus route from north side often runs late in morning traffic'

In [50]:
def calculate_auto_fill_score(student_id, session_num, llm, student_name, is_first_entry):
    """Determining the score based on circumstances and stuff"""

    student_circumstances = get_student_circumstances(student_id, session_num)
    if is_first_entry:
        # Check circumstances for sessions BEFORE current ( the first entry)
        ai_prompt = f"""
        AUTO-FILL SCORING FOR FIRST ENTRY:
        
        STUDENT: {student_name}
        SESSION: {session_num} (session being auto-filled)
        FIRST ENTRY: TRUE
        CIRCUMSTANCES: {student_circumstances}
        
        SCORING RULES FOR SESSIONS BEFORE FIRST ENTRY:
        - If student has 'full' excuse for this session: score = 1.0
        - If student has 'partial' or 'late_arrival' excuse: score = 0.5  
        - If no valid excuse: score = 0.0 (absent)
        
        Analyze the circumstances and return: score,reason
        Examples:
        1.0,has_medical_excuse_for_this_session
        0.5,has_transportation_issues_for_morning_sessions
        0.0,no_documented_excuse_for_this_session
        
        Your decision:
        """
    else:
        #  Between last entry and current exit
        ai_prompt = f"""
        AUTO-FILL SCORING FOR MISSED SESSION:
        
        STUDENT: {student_name}
        SESSION: {session_num} (session being auto-filled)
        CIRCUMSTANCES: {student_circumstances}
        
        SCORING RULES FOR MISSED SESSIONS BETWEEN RECORDED ENTRIES:
        - Assume student was present but forgot to record entry
        - Always give score = 1.0
        - Only deduct if there's clear evidence they were absent
        
        Return: score,reason
        Examples:
        1.0,assumed_present_between_recorded_sessions
        1.0,student_likely_present_based_on_movement_pattern
        0.0,clear_evidence_of_absence_from_circumstances
        
        Your decision:
        """
    
    ai_response = llm.invoke(ai_prompt).content.strip()
    score, reason = ai_response.split(',')
    
    return {
        'score': float(score),
        'reason': reason
    }
    


In [54]:
def auto_fill_missing_sessions(student_id, current_session_num, llm, student_name, is_first_entry):
    """ Auto fill the session before the first entry
    and the session between the last entry and the nearest exist"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            filled_sessions = []
            current_date = current_datetime.strftime('%Y-%m-%d')
            #Get the last session_num from database
            cursor.execute("""
                SELECT MAX(CAST(session_number AS INTEGER)) 
                FROM attendance_sessions 
                WHERE student_id = ? AND session_date = ?
            """, (student_id, current_date))
            result = cursor.fetchone()
            #If none session, default to 0
            last_session_num = result[0] if result[0] else 0
            for session_num in range(last_session_num +1, current_session_num):
                session_info = get_session_by_number(session_num)
                if session_info:
                    auto_fill_score = calculate_auto_fill_score(student_id,session_num,
                                                                llm, student_name, is_first_entry)
                    #Marking present for the missed session
                    cursor.execute("""
                        INSERT INTO attendance_sessions 
                        (student_id, session_date, entry_time, status, attendance_status, 
                         session_number, reason_for_scoring, late_minutes)
                        VALUES (?, ?, ?, 'present', 'on_time', ?, 
                               ?, 0)
                    """, (
                        student_id, current_date, 
                        datetime.now().strftime('%H:%M:%S'),
                        session_num,
                        f"AUTO_FILLED: {auto_fill_score['reason']} (score:{auto_fill_score['score']})"
                    ))                    
                    filed_sessions.append({
                        'session':session_num,
                        'score':auto_fill_score['score'],
                        'reason':auto_fill_score['reason']
                    })

                    logging.info(f"Auto-filled session {session_num} for {student_name}, ID: {student_id} with score {auto_fill_score['score']} ")
            
            conn.comit()
            return filled_sessions

    except Exception as e:
        logging.error(f"Auto fill score failed: {e}")
        return []
            